# Lab 5 — Cleaning and Transforming Data in Pandas

**Course:** Python for AI & Data  
**Analyst:** *Natalia Amaya*  
**Date:** *4/20/2026*

---

## The PaperLane Scenario

PaperLane's sales team has been logging orders manually across spreadsheets. The data has never been cleaned. Before any analysis can happen, it needs to be fixed.

Your job: take the raw order export and produce a clean, analysis-ready dataset.

Work from top to bottom. Run every cell as you go.


---
## Task 1 — Load and First Look

Load the raw data from `data/raw/paperlane_orders.csv` into a DataFrame.

After loading:
- Display the first 5 rows with `head()`
- Print the shape (rows, columns)
- Print the column names


In [9]:
import pandas as pd
from pathlib import Path

# Base project folder (go up from notebooks)
BASE_DIR = Path.cwd().parent

RAW = BASE_DIR / "data" / "raw"
PROCESSED = BASE_DIR / "data" / "processed"


# Load the raw data
file_path = RAW / "paperlane_orders.csv"
df = pd.read_csv(file_path)

# Display first 5 rows

print(df.head())

# Print shape and column names

print(df.shape)
print(df.columns)


   order_id  order_date customer   region   product  quantity  unit_price  \
0      1001  2026-01-05      Ava     East  Notebook         2        8.99   
1      1002  2026-01-05    Miles  Midwest       Pen        10        1.25   
2      1003  2026-01-06     Noah     West  Notebook         1        8.99   
3      1004  2026-01-06      Ava     East  Backpack         1         NaN   
4      1005  2026-01-07    Priya    South       Pen         5        1.25   

    channel    status  
0    Online  Complete  
1  In-Store  Complete  
2    Online  Complete  
3    Online  Returned  
4  In-Store  Complete  
(21, 9)
Index(['order_id', 'order_date', 'customer', 'region', 'product', 'quantity',
       'unit_price', 'channel', 'status'],
      dtype='str')


---
## Task 2 — Structured Inspection

Before touching the data, build a complete picture of its quality issues.

Use all of the following:
- `df.info()` — types and null counts
- `df.isna().sum()` — missing values per column
- `df.duplicated().sum()` — duplicate rows
- `df["status"].value_counts()` — spot inconsistent values
- `df["region"].value_counts()` — spot whitespace issues

Then, in the markdown cell below the code, **document every issue you found** before making any changes.


In [10]:
# Run all inspection methods here
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    21 non-null     int64  
 1   order_date  21 non-null     str    
 2   customer    21 non-null     str    
 3   region      19 non-null     str    
 4   product     21 non-null     str    
 5   quantity    21 non-null     int64  
 6   unit_price  19 non-null     float64
 7   channel     21 non-null     str    
 8   status      21 non-null     str    
dtypes: float64(1), int64(2), str(6)
memory usage: 1.6 KB


Region and Unit_Price are missing 2 values. Order_date is a string. 

In [11]:
df.isna().sum()

order_id      0
order_date    0
customer      0
region        2
product       0
quantity      0
unit_price    2
channel       0
status        0
dtype: int64

There are N/A values in region and unit_price

In [12]:
df.duplicated().sum()

np.int64(1)

There is one duplicate row

In [13]:
df["status"].value_counts() 

status
Complete    16
COMPLETE     2
Returned     1
complete     1
returned     1
Name: count, dtype: int64

There are 3 different values for the status "Complete" and 2 for "Returned"

In [14]:
df["region"].value_counts()

region
East        5
Midwest     4
South       3
West        3
 West       2
 East       1
Midwest     1
Name: count, dtype: int64

There are whitespaces separating South, East, West and Midwest into different values. 

In [15]:
df

,order_id,order_date,customer,region,product,quantity,unit_price,channel,status
0,1001,2026-01-05,Ava,East,Notebook,2,8.99,Online,Complete
1,1002,2026-01-05,Miles,Midwest,Pen,10,1.25,In-Store,Complete
2,1003,2026-01-06,Noah,West,Notebook,1,8.99,Online,Complete
3,1004,2026-01-06,Ava,East,Backpack,1,NaN,Online,Returned
4,1005,2026-01-07,Priya,South,Pen,5,1.25,In-Store,Complete
5,1006,2026-01-07,Noah,West,Marker,3,2.50,In-Store,COMPLETE
6,1007,2026-01-08,Miles,Midwest,Backpack,2,39.00,Online,Complete
7,1007,2026-01-08,Miles,Midwest,Backpack,2,39.00,Online,Complete
8,1008,2026-01-08,Ava,NaN,Marker,4,2.50,In-Store,Complete
9,1009,01/09/2026,Priya,South,Notebook,2,8.99,Online,Complete


### Data Quality Issues Found

This data frame has 2 missing values in unit_price, 2 duplicate rows, whitespaces affecting region values and several ways status values were recorded. 


---
## Task 3 — Remove Duplicates

Remove any duplicate rows from the DataFrame.

Print the shape before and after to confirm how many rows were removed.


In [16]:
# Print shape before
print(df.shape)

(21, 9)


In [17]:
# Remove duplicates
df = df.drop_duplicates()

In [18]:
# Print shape after
print(df.shape)

(20, 9)


---
## Task 4 — Handle Missing Values

Two columns have missing values. Handle each one.

For each column:
1. Choose a strategy (fill with a value, fill with a statistic, or drop)
2. Apply it
3. In the markdown cell below, explain why you chose that strategy

After handling both, verify with `df.isna().sum()` — all values should be 0.


In [21]:
# Handle missing values
avg_price = df["unit_price"].mean()

df["unit_price"] = df["unit_price"].fillna(avg_price)


In [22]:
# Verify all missing values are resolved
print(df["unit_price"].isnull().sum())

0


### Cleaning Decisions

I chose to fill N/As with a mean instead of removing entire rows to preserve other important data included.

---
## Task 5 — Standardise Text Columns

Fix inconsistent casing and whitespace in `status`, `region`, and `channel`.

Use `str.strip()` to remove whitespace and `str.title()` to normalise casing.

After each fix, run `value_counts()` to confirm the column is now consistent.


In [23]:
# Standardise status
df["status"] = df["status"].str.strip().str.lower()

# Standardise region
df["region"] = df["region"].str.strip().str.title()

# Standardise channel
df["channel"] = df["channel"].str.strip().str.title()

# Verify each column
df["status"].value_counts()


status
complete    18
returned     2
Name: count, dtype: int64

---
## Task 6 — Convert the Date Column

The `order_date` column currently has `object` dtype and contains some non-standard date formats.

Convert it to a proper `datetime` type using `pd.to_datetime()`.

After converting, confirm the dtype has changed by printing `df["order_date"].dtype`.


In [26]:
# Convert order_date to datetime

df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")

# Confirm the dtype
print(df["order_date"].dtype)


datetime64[us]


---
## Task 7 — Create Derived Columns

Add two new columns to the DataFrame:

1. **`revenue`** = `quantity` × `unit_price`
2. **`order_month`** = the month number extracted from `order_date` (hint: use `.dt.month`)

Display a sample of the relevant columns to confirm.


In [27]:
# Create revenue column

df["revenue"] = df["quantity"] * df["unit_price"]

# Create order_month column

df["order_month"] = df["order_date"].dt.month


# Display a sample

df[["order_date", "order_month", "quantity", "unit_price", "revenue"]].head()


,order_date,order_month,quantity,unit_price,revenue
0,2026-01-05,1,2,8.990000,17.980000
1,2026-01-05,1,10,1.250000,12.500000
2,2026-01-06,1,1,8.990000,8.990000
3,2026-01-06,1,1,10.178333,10.178333
4,2026-01-07,1,5,1.250000,6.250000


---
## Task 8 — Filter and Sort

Create a new DataFrame called `df_complete` containing only orders with `status == "Complete"`.

- Use `.copy()` when creating the filtered DataFrame
- Sort by `revenue` descending
- Print how many complete orders there are out of the total
- Display the top 5 rows


In [32]:
# Filter to complete orders only
df_complete = df[df["status"] == "complete"].copy()

# Sort by revenue descending
df_complete = df_complete.sort_values("revenue", ascending=False)

# Print count and display top 5
print("Complete orders:", len(df_complete))

df_complete.head()



Complete orders: 18


,order_id,order_date,customer,region,product,quantity,unit_price,channel,status,revenue,order_month
6,1007,2026-01-08,Miles,Midwest,Backpack,2,39.000000,Online,complete,78.000000,1
12,1012,2026-01-10,Sofia,West,Backpack,1,39.000000,Online,complete,39.000000,1
17,1017,2026-01-13,Priya,South,Backpack,1,39.000000,In-Store,complete,39.000000,1
18,1018,2026-01-13,Miles,Midwest,Notebook,3,8.990000,Online,complete,26.970000,1
13,1013,2026-01-11,Ava,East,Binder,2,10.178333,In-Store,complete,20.356667,1


---
## Task 9 — Save Cleaned Data

Save two files to `data/processed/`:

1. The full cleaned DataFrame → `paperlane_orders_clean.csv`
2. The complete-orders DataFrame → `paperlane_orders_complete.csv`

Use `index=False` in `to_csv()`. Create the `processed/` folder if it doesn't exist.

After saving, verify both files exist using `Path.exists()`.


In [37]:
# Create processed directory
PROCESSED.mkdir(parents=True, exist_ok=True)

# Save full cleaned DataFrame
df.to_csv(PROCESSED / "paperlane_orders_clean.csv", index=False)

# Save complete-orders DataFrame

df_complete.to_csv(PROCESSED / "paperlane_orders_complete.csv", index=False)

# Verify both files exist

clean_file = PROCESSED_DIR / "paperlane_orders_clean.csv"
complete_file = PROCESSED_DIR / "paperlane_orders_complete.csv"

print("Clean file exists:", clean_file.exists())
print("Complete file exists:", complete_file.exists())

NameError: name 'PROCESSED_DIR' is not defined

---
## ⭐ Optional Advanced Tasks

### Optional 1 — Revenue by Region
Group `df_complete` by `region` and calculate total revenue, number of orders, and average order value per region. Display as a DataFrame sorted by total revenue descending.

### Optional 2 — Revenue by Channel
Do the same grouping but by `channel`. Does Online or In-Store drive more revenue for PaperLane?

### Optional 3 — Pivot Table
Build a pivot table showing total revenue by `region` (rows) and `product` (columns). Use `pd.pivot_table()`. Fill missing combinations with 0.


In [ ]:
# Optional tasks — work here
